# 29. Unsupervised Learning: t-SNE (t-Distributed Stochastic Neighbor Embedding)

## Algorithm Category
**Type**: Unsupervised Learning - Dimensionality Reduction & Visualization  
**Complexity**: Medium-High  
**Use Case**: Non-linear dimensionality reduction for visualization of high-dimensional data

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand t-SNE and its mathematical foundation
- Implement t-SNE for data visualization
- Understand perplexity and its role in t-SNE
- Compare t-SNE with PCA
- Visualize high-dimensional data in 2D/3D
- Apply t-SNE to real-world problems

## Historical Context

t-SNE was developed by Laurens van der Maaten and Geoffrey Hinton in 2008:
- van der Maaten, L. & Hinton, G. (2008): "Visualizing Data using t-SNE"
- Extension of SNE (Stochastic Neighbor Embedding)
- Widely used for visualizing high-dimensional data

**Key Papers/References:**
- van der Maaten, L. & Hinton, G. (2008). "Visualizing Data using t-SNE"
- van der Maaten, L. (2014). "Accelerating t-SNE using Tree-Based Algorithms"

## When to Use t-SNE

t-SNE is appropriate when:
- You need to visualize high-dimensional data
- Data has non-linear structure
- You want to explore cluster structure
- Working with embeddings or feature vectors
- Need to understand data relationships
- Data has local structure to preserve

## Theory & Mechanics

### Mathematical Foundation

t-SNE preserves local neighborhood structure by modeling pairwise similarities.

**High-Dimensional Space (Gaussian):**
$$p_{j|i} = \frac{\exp(-||x_i - x_j||^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-||x_i - x_k||^2 / 2\sigma_i^2)}$$

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2N}$$

**Low-Dimensional Space (t-Distribution):**
$$q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum_{k \neq l} (1 + ||y_k - y_l||^2)^{-1}}$$

**Cost Function (KL Divergence):**
$$C = \sum_{i} \sum_{j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Key Differences from SNE:**
- Uses symmetric joint probabilities
- Uses t-distribution (heavy-tailed) in low-dimensional space
- Better at preserving global structure

### How It Works

1. **Compute similarities**: Calculate pairwise similarities in high-dimensional space
2. **Set perplexity**: Choose perplexity (related to number of neighbors)
3. **Initialize**: Random initialization in low-dimensional space
4. **Optimize**: Minimize KL divergence using gradient descent
5. **Iterate**: Update positions until convergence

### Key Hyperparameters

- **n_components**: Number of dimensions for embedding (usually 2 or 3)
- **perplexity**: Effective number of neighbors (typically 5-50)
  - Lower: Focus on local structure
  - Higher: Focus on global structure
- **learning_rate**: Step size for optimization (default: 200)
- **n_iter**: Maximum iterations (default: 1000)
- **random_state**: Seed for reproducibility

### Advantages

- Captures non-linear structure
- Excellent for visualization
- Preserves local neighborhoods
- Can reveal cluster structure
- Works well with high-dimensional data

### Limitations

- Computationally expensive (O(n²))
- Non-deterministic (results vary)
- Cannot be applied to new data (no transform)
- Sensitive to perplexity
- May not preserve global structure
- Slow for large datasets


## Implementation

Let's implement t-SNE for data visualization.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Libraries imported successfully!")


In [ ]:
# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print(f"Dataset Shape: {X.shape}")
print(f"Features: {iris.feature_names}")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

print(f"\nt-SNE Results:")
print(f"  Reduced shape: {X_tsne.shape}")
print(f"  Perplexity: {tsne.perplexity}")
print(f"  KL divergence: {tsne.kl_divergence_:.3f}")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
# Original data (first 2 features)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Original Data (First 2 Features)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Projection (2D)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Effect of Perplexity

Let's see how perplexity affects t-SNE results.


In [ ]:
# Test different perplexity values
perplexities = [5, 15, 30, 50]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, perp in enumerate(perplexities):
    row = idx // 2
    col = idx % 2
    
    tsne_perp = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_tsne_perp = tsne_perp.fit_transform(X_scaled)
    
    axes[row, col].scatter(X_tsne_perp[:, 0], X_tsne_perp[:, 1], c=y, 
                          cmap='viridis', s=50, alpha=0.7)
    axes[row, col].set_title(f'Perplexity = {perp}\nKL Divergence: {tsne_perp.kl_divergence_:.2f}')
    axes[row, col].set_xlabel('t-SNE Component 1')
    axes[row, col].set_ylabel('t-SNE Component 2')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Effect of Perplexity:")
for perp in perplexities:
    tsne_perp = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_tsne_perp = tsne_perp.fit_transform(X_scaled)
    print(f"  Perplexity {perp}: KL Divergence = {tsne_perp.kl_divergence_:.3f}")


## Comparison with PCA

Let's compare t-SNE with PCA.


In [ ]:
# Apply PCA for comparison
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Visualize comparison
plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('Original Data\n(First 2 Features)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.title('PCA Projection\n(Linear)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Projection\n(Non-linear)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Comparison:")
print(f"  PCA: Linear dimensionality reduction")
print(f"  t-SNE: Non-linear dimensionality reduction")
print(f"  Note: t-SNE better preserves local structure and clusters")


## Validation & Testing

Let's validate t-SNE and check for consistency.


In [ ]:
# Test reproducibility (with same random_state)
tsne1 = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne1 = tsne1.fit_transform(X_scaled)

tsne2 = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne2 = tsne2.fit_transform(X_scaled)

# Check if results are identical (they should be with same random_state)
are_identical = np.allclose(X_tsne1, X_tsne2)
print(f"Results with same random_state are identical: {are_identical}")

# Test with different random_state (results will differ)
tsne3 = TSNE(n_components=2, perplexity=30, random_state=123, n_iter=1000)
X_tsne3 = tsne3.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_tsne1[:, 0], X_tsne1[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.title('t-SNE (random_state=42)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_tsne3[:, 0], X_tsne3[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
plt.title('t-SNE (random_state=123)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: t-SNE results vary with different random_state, but cluster structure should be similar")

# Assertions
assert X_tsne.shape[1] == 2, "Should have 2 components"
assert tsne.kl_divergence_ > 0, "KL divergence should be positive"
print("\n✓ Validation checks passed")


## Real-World Application

Let's apply t-SNE to a higher-dimensional dataset (Digits).


In [ ]:
# Load Digits dataset (64 dimensions)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits Dataset Shape: {X_digits.shape}")
print(f"Number of features: {X_digits.shape[1]}")

# Standardize
scaler_digits = StandardScaler()
X_digits_scaled = scaler_digits.fit_transform(X_digits)

# Apply t-SNE (use PCA first for speed with high-dimensional data)
print("\nApplying PCA first (for speed), then t-SNE...")
pca_digits = PCA(n_components=50, random_state=42)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)

tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_digits_tsne = tsne_digits.fit_transform(X_digits_pca)

print(f"t-SNE Results:")
print(f"  Reduced shape: {X_digits_tsne.shape}")
print(f"  KL divergence: {tsne_digits.kl_divergence_:.3f}")

# Visualize
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
# Show sample digits
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(digits.images[i], cmap='gray')
    plt.title(f'Digit {digits.target[i]}')
    plt.axis('off')
plt.suptitle('Sample Digits', y=1.02)
plt.tight_layout()

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], c=y_digits, 
                     cmap='tab10', s=30, alpha=0.6)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('Digits Dataset: t-SNE Projection (2D)')
plt.colorbar(scatter, label='Digit')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: t-SNE reveals clear separation between digit classes")


## Summary & Key Takeaways

### Key Concepts Learned

1. **t-SNE Basics**
   - Non-linear dimensionality reduction
   - Preserves local neighborhood structure
   - Uses t-distribution in low-dimensional space
   - Excellent for visualization

2. **Mathematical Foundation**
   - Models pairwise similarities in high-D (Gaussian)
   - Models pairwise similarities in low-D (t-distribution)
   - Minimizes KL divergence between distributions
   - Gradient descent optimization

3. **Key Hyperparameters**
   - **perplexity**: Number of effective neighbors (5-50)
   - **n_components**: Output dimensions (usually 2 or 3)
   - **learning_rate**: Optimization step size
   - **n_iter**: Maximum iterations

4. **Best Practices**
   - Use PCA first for high-dimensional data (speed)
   - Standardize features before t-SNE
   - Experiment with perplexity (start with 30)
   - Use random_state for reproducibility
   - Results are non-deterministic (vary with initialization)

### When to Use t-SNE

✅ **Good for:**
- Visualizing high-dimensional data
- Exploring cluster structure
- Non-linear data relationships
- Understanding data topology
- Embedding visualization
- Small to medium datasets (< 10,000 samples)

❌ **Not ideal for:**
- Very large datasets (computationally expensive O(n²))
- When you need to transform new data (no transform method)
- When global structure is important
- When deterministic results are required
- Real-time applications (slow)
- When linear methods (PCA) suffice

### Next Steps

- Compare with **UMAP** (faster alternative)
- Use **PCA + t-SNE** for high-dimensional data
- Try **3D t-SNE** for better visualization
- Explore **Barnes-Hut t-SNE** for faster computation
- Apply to **word embeddings** and **image features**
